# 1. Data, features and split

We use one-minute historical Binance candles and documented event times. The project code creates all features and labels; this notebook shows the main checks.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "pumpwatch" / "train.py").exists():
    ROOT = ROOT.parent
assert (ROOT / "pumpwatch" / "train.py").exists(), "Open from the project root or notebooks folder"
sys.path.insert(0, str(ROOT))
import pandas as pd
from pumpwatch.data import DATA
from pumpwatch.features import FEATURES, engineer, label_windows
from pumpwatch.train import split_by_time
candles = pd.read_csv(DATA / "candles.csv.gz", parse_dates=["timestamp"])
events = pd.read_csv(DATA / "events.csv")
announcements = pd.read_csv(DATA / "known_announcements.csv")
print(f"Candles: {len(candles):,}; events: {len(events):,}; pairs: {candles.symbol.nunique()}")
display(candles.head(3))


## Features and labels

Returns, volume anomalies, volatility and candle shape use the current and earlier candles. The rolling volume and trade baselines are shifted by one minute. A prediction is available when the candle closes.

In [ ]:
frame = label_windows(engineer(candles), events, announcements)
print("Features:", ", ".join(FEATURES))
display(frame.loc[frame.ready, ["timestamp", "symbol", "prediction_time", "target"] + FEATURES[:6]].head())
print("Labels: 1 = onset and following two minutes, 0 = no known event, -1 = excluded vicinity")
display(frame.target.value_counts().sort_index().rename("windows").to_frame())


## Chronological holdout

We use earlier event dates for training, then validation, then test. Sixty minutes around each split boundary are purged. Only training negatives are sampled later; validation and test keep their eligible negatives.

In [ ]:
frame, boundaries = split_by_time(frame, events)
usable = frame[frame.ready & (frame.target >= 0) & (frame.split != "purged")]
summary = usable.groupby("split").agg(windows=("target", "size"), positives=("target", "sum"), first=("timestamp", "min"), last=("timestamp", "max"))
display(summary.loc[["train", "validation", "test"]])
print(boundaries)


**Presentation note:** We label the onset and next two minutes, so the model recognizes the start of a recorded event. It does not establish market manipulation or forecast an event before it starts.